In [4]:
import pandas as pd
import os

raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

# 1. Pulizia EDGES
print("Processing edges...")
edges = pd.read_csv(os.path.join(raw_path, 'edges.csv'))
# Rinominiamo le colonne Neo4j in standard Python
edges = edges.rename(columns={':START_ID': 'source', ':END_ID': 'target', ':TYPE': 'type'})
edges.to_csv(os.path.join(proc_path, 'edges_cleaned.csv'), index=False)

Processing edges...


In [3]:
import pandas as pd
import os

raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

# 1. Caricamento del file originale
# Nota: sostituisci 'nodes.csv' con il nome esatto del tuo file nella cartella raw
raw_nodes = pd.read_csv(os.path.join(raw_path, 'nodes.csv'), low_memory=False)

# 2. Pulizia Nomi Colonne (rimuoviamo i suffissi Neo4j)
raw_nodes.columns = [c.split(':')[0].strip() for c in raw_nodes.columns]
raw_nodes.columns = [c.lstrip(':') for c in raw_nodes.columns]

# 3. Definizione delle colonne da mantenere
# Mappiamo le colonne originali a nomi più semplici e puliti
cols_map = {
    'celex_id': 'celex',
    'work_title': 'title',
    'year': 'year',
    'domains': 'domains',
    'subdomains': 'subdomains',
    'resource_legal_type': 'legal_type',
    'url': 'url'
}

# Se 'celex_id' non esiste, cerchiamo 'id' come fallback
if 'celex_id' not in raw_nodes.columns and 'id' in raw_nodes.columns:
    raw_nodes = raw_nodes.rename(columns={'id': 'celex_id'})

# Selezioniamo solo le colonne che esistono effettivamente nel file raw
existing_cols = [c for c in cols_map.keys() if c in raw_nodes.columns]
nodes_cleaned = raw_nodes[existing_cols].rename(columns=cols_map)

# 4. Filtro Nodi spuri (quelli senza CELEX o con ID alfanumerici casuali)
# Teniamo solo le righe dove celex non è nullo e non è la stringa 'N/A'
nodes_cleaned = nodes_cleaned[nodes_cleaned['celex'].notna() & (nodes_cleaned['celex'] != 'N/A')]

# 5. Fix per il Titolo (work_title)
# Se il titolo è vuoto o NaN, usiamo il CELEX come titolo di emergenza
nodes_cleaned['title'] = nodes_cleaned['title'].fillna(nodes_cleaned['celex'])

# 6. Pulizia stringhe EuroVoc (rimozione di parentesi e virgolette residue)
for col in ['domains', 'subdomains']:
    if col in nodes_cleaned.columns:
        nodes_cleaned[col] = nodes_cleaned[col].astype(str).str.replace(r'[\[\]"]', '', regex=True)
        # Se dopo la pulizia la stringa è 'nan', mettiamo il nostro default
        nodes_cleaned[col] = nodes_cleaned[col].replace('nan', '00 UNKNOWN')

# 7. Salvataggio finale
nodes_cleaned.to_csv(os.path.join(proc_path, 'nodes_cleaned.csv'), index=False)
print(f"Dataset normalizzato con successo! {len(nodes_cleaned)} nodi salvati in 'nodes_cleaned.csv'.")

Dataset normalizzato con successo! 86357 nodi salvati in 'nodes_cleaned.csv'.


In [3]:
import pandas as pd
import os

# 1. Definizione percorsi (usiamo i percorsi relativi alla radice del progetto)
raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

def clean_headers(df):
    """Rimuove i suffissi Neo4j dai nomi delle colonne (es. id:ID -> id)"""
    new_cols = []
    for col in df.columns:
        clean_name = col.split(':')[0] if ':' in col else col
        # Gestisce i casi tipo :LABEL o :TYPE (rimuove il due punti iniziale)
        if not clean_name and ':' in col:
            clean_name = col.split(':')[1]
        new_cols.append(clean_name.lower())
    df.columns = new_cols
    return df

print("--- Inizio Normalizzazione Dataset ---")

# --- A. Pulizia NODES (Gestione NaN definitiva) ---
nodes_file = os.path.join(proc_path, 'nodes_cleaned.csv')
if os.path.exists(nodes_file):
    nodes = pd.read_csv(nodes_file)
    nodes['work_title'] = nodes['work_title'].fillna('Titolo non disponibile')
    nodes['domains'] = nodes['domains'].fillna('00 UNKNOWN')
    nodes['celex'] = nodes['celex'].fillna('N/A')
    nodes.to_csv(nodes_file, index=False)
    print(f"1. nodes_cleaned.csv: NaN risolti per {len(nodes)} righe.")

# --- B. Pulizia EUROVOC CONCEPTS (Dizionario) ---
concepts_raw = os.path.join(raw_path, 'eurovoc_concept.csv')
if os.path.exists(concepts_raw):
    concepts = pd.read_csv(concepts_raw)
    concepts = clean_headers(concepts)
    # Rimuoviamo le parentesi quadre STRING[] dai dati se presenti
    for col in ['eurovoc_concepts', 'domains', 'subdomains']:
        if col in concepts.columns:
            concepts[col] = concepts[col].str.replace(r'[\[\]"]', '', regex=True)
    concepts.to_csv(os.path.join(proc_path, 'eurovoc_concepts_cleaned.csv'), index=False)
    print("2. eurovoc_concepts_cleaned.csv: Creato (nomi colonne puliti).")

# --- C. Pulizia ARCHI (has_concept e parent_of) ---
for f_name in ['has_concept_edges.csv', 'parent_of.csv']:
    f_path = os.path.join(raw_path, f_name)
    if os.path.exists(f_path):
        df = pd.read_csv(f_path)
        # Rinominiamo i :START_ID e :END_ID in source e target
        df.columns = ['source', 'target', 'type']
        out_name = f_name.replace('.csv', '_cleaned.csv')
        df.to_csv(os.path.join(proc_path, out_name), index=False)
        print(f"3. {out_name}: Creato con colonne standard (source, target).")

print("--- Normalizzazione Completata! ---")

--- Inizio Normalizzazione Dataset ---
1. nodes_cleaned.csv: NaN risolti per 88130 righe.
2. eurovoc_concepts_cleaned.csv: Creato (nomi colonne puliti).
3. has_concept_edges_cleaned.csv: Creato con colonne standard (source, target).
3. parent_of_cleaned.csv: Creato con colonne standard (source, target).
--- Normalizzazione Completata! ---


In [6]:
import pandas as pd
import os

proc_path = os.path.join('..', 'data', 'processed')

# Carichiamo i file necessari
nodes = pd.read_csv(os.path.join(proc_path, 'nodes_cleaned.csv'))
concepts = pd.read_csv(os.path.join(proc_path, 'eurovoc_concepts_cleaned.csv'))
edges_concept = pd.read_csv(os.path.join(proc_path, 'has_concept_edges_cleaned.csv'))

# 1. Parsing del CELEX per Year e Legal Type
def parse_celex(celex):
    if pd.isna(celex) or len(celex) < 7: return None, None
    year = celex[1:5] # Estrae l'anno (posizioni 1-4)
    l_type = celex[5] # Estrae il tipo (posizione 5)
    
    type_map = {'R': 'Regulation', 'L': 'Directive', 'D': 'Decision', 'C': 'Communication'}
    return year, type_map.get(l_type, 'Other')

nodes[['year', 'legal_type']] = nodes['celex'].apply(lambda x: pd.Series(parse_celex(str(x))))

# 2. Back-filling dei DOMAINS tramite gli archi
# Creiamo una mappa Concept -> Domain dal file eurovoc_concepts
concept_to_domain = concepts.set_index('id')['domains'].to_dict()

# Troviamo i domini per ogni legge tramite gli archi
edges_concept['domain_resolved'] = edges_concept['target'].map(concept_to_domain)

# Raggruppiamo i domini trovati per ogni legge (source)
law_domains = edges_concept.groupby('source')['domain_resolved'].first().to_dict()

# Riempiamo i domini mancanti nei nodi
nodes['domains'] = nodes['domains'].replace('00 UNKNOWN', pd.NA)
nodes['domains'] = nodes['domains'].fillna(nodes['celex'].map(law_domains))
nodes['domains'] = nodes['domains'].fillna('00 UNKNOWN')

# 3. Salvataggio finale arricchito
nodes.to_csv(os.path.join(proc_path, 'nodes_cleaned.csv'), index=False)
print("Dati arricchiti! Year, Type e Domains recuperati dove possibile.")

Dati arricchiti! Year, Type e Domains recuperati dove possibile.
